In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/database/flights.sqlite")

conn = sqlite3.connect(DB_PATH)

BASE_FEATURES = """
    Year,
    Month,
    DayofMonth,
    DayOfWeek,
    FlightDate,
    Reporting_Airline,
    Flight_Number_Reporting_Airline,
    Origin,
    Dest,
    CRSDepTime,
    CRSDepHour,
    CRSArrTime,
    CRSArrHour,
    CRSElapsedTime,
    Distance,
    DistanceGroup
"""

In [2]:
cancelled_query = f"""
SELECT
    {BASE_FEATURES},
    CAST(Cancelled AS INTEGER) AS target_cancelled
FROM raw_ontime_flights
WHERE Cancelled IS NOT NULL
  AND rowid % 10 = 0;
"""

cancelled_df = pd.read_sql_query(cancelled_query, conn)

print(cancelled_df.shape)
print(cancelled_df["target_cancelled"].value_counts())
print(cancelled_df["target_cancelled"].value_counts(normalize=True))

(2092857, 17)
target_cancelled
0    2064242
1      28615
Name: count, dtype: int64
target_cancelled
0    0.986327
1    0.013673
Name: proportion, dtype: float64


In [3]:
delayed_query = f"""
SELECT
    {BASE_FEATURES},
    CAST(DepDel15 AS INTEGER) AS target_delayed
FROM raw_ontime_flights
WHERE Cancelled = 0
  AND DepDel15 IS NOT NULL
  AND rowid % 10 = 0;
"""

delayed_df = pd.read_sql_query(delayed_query, conn)

print(delayed_df.shape)
print(delayed_df["target_delayed"].value_counts())
print(delayed_df["target_delayed"].value_counts(normalize=True))

(2064242, 17)
target_delayed
0    1631415
1     432827
Name: count, dtype: int64
target_delayed
0    0.790322
1    0.209678
Name: proportion, dtype: float64


In [4]:
diverted_query = f"""
SELECT
    {BASE_FEATURES},
    CAST(Diverted AS INTEGER) AS target_diverted
FROM raw_ontime_flights
WHERE Cancelled = 0
  AND Diverted IS NOT NULL
  AND rowid % 10 = 0;
"""

diverted_df = pd.read_sql_query(diverted_query, conn)

print(diverted_df.shape)
print(diverted_df["target_diverted"].value_counts())
print(diverted_df["target_diverted"].value_counts(normalize=True))

(2064242, 17)
target_diverted
0    2059037
1       5205
Name: count, dtype: int64
target_diverted
0    0.997478
1    0.002522
Name: proportion, dtype: float64


In [5]:
for name, df, target in [("Cancellation", cancelled_df, "target_cancelled"), ("Delay", delayed_df, "target_delayed"), ("Diversion", diverted_df, "target_diverted"),]:
    print(df[target].value_counts(dropna=False))
    print(df[target].value_counts(normalize=True, dropna=False).round(4))

target_cancelled
0    2064242
1      28615
Name: count, dtype: int64
target_cancelled
0    0.9863
1    0.0137
Name: proportion, dtype: float64
target_delayed
0    1631415
1     432827
Name: count, dtype: int64
target_delayed
0    0.7903
1    0.2097
Name: proportion, dtype: float64
target_diverted
0    2059037
1       5205
Name: count, dtype: int64
target_diverted
0    0.9975
1    0.0025
Name: proportion, dtype: float64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest"
]

numeric_features = [
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "Flight_Number_Reporting_Airline",
    "CRSDepTime",
    "CRSDepHour",
    "CRSArrTime",
    "CRSArrHour",
    "CRSElapsedTime",
    "Distance",
    "DistanceGroup"
]

model_features = categorical_features + numeric_features

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=10
        )
    )
])

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    ),
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    )
])

def make_baseline_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                max_iter=1000,
                tol=1e-3,
                random_state=42
            )
        )
    ])

In [8]:
def train_and_evaluate(df, target_column, model_name):
    train = df[df["Year"].isin([2023, 2024])].copy()
    test = df[df["Year"] == 2025].copy()

    X_train = train[model_features]
    y_train = train[target_column].astype(int)

    X_test = test[model_features]
    y_test = test[target_column].astype(int)

    model = make_baseline_model()
    model.fit(X_train, y_train)

    predicted_probability = model.predict_proba(X_test)[:, 1]
    predictions = (predicted_probability >= 0.5).astype(int)

    prevalence = y_test.mean()
    pr_auc = average_precision_score(
        y_test,
        predicted_probability
    )
    roc_auc = roc_auc_score(
        y_test,
        predicted_probability
    )

    print(f"\n{model_name}")
    print("=" * len(model_name))
    print(f"Training rows: {len(train):,}")
    print(f"Testing rows:  {len(test):,}")
    print(f"Positive test rate: {prevalence:.4%}")
    print(f"Baseline PR AUC: {prevalence:.4f}")
    print(f"Model PR AUC:    {pr_auc:.4f}")
    print(f"ROC AUC:         {roc_auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            predictions,
            digits=4,
            zero_division=0
        )
    )

    return {
        "name": model_name,
        "model": model,
        "target": target_column,
        "test_prevalence": prevalence,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "X_test": X_test,
        "y_test": y_test,
        "probabilities": predicted_probability
    }

In [9]:
cancelled_results = train_and_evaluate(
    cancelled_df,
    target_column="target_cancelled",
    model_name="Cancellation model"
)

delayed_results = train_and_evaluate(
    delayed_df,
    target_column="target_delayed",
    model_name="Departure-delay model"
)

diverted_results = train_and_evaluate(
    diverted_df,
    target_column="target_diverted",
    model_name="Diversion model"
)


Cancellation model
Training rows: 1,392,696
Testing rows:  700,161
Positive test rate: 1.4584%
Baseline PR AUC: 0.0146
Model PR AUC:    0.0243
ROC AUC:         0.6349

Confusion matrix:
[[437158 252792]
 [  4380   5831]]

Classification report:
              precision    recall  f1-score   support

           0     0.9901    0.6336    0.7727    689950
           1     0.0225    0.5711    0.0434     10211

    accuracy                         0.6327    700161
   macro avg     0.5063    0.6023    0.4080    700161
weighted avg     0.9760    0.6327    0.7621    700161


Departure-delay model
Training rows: 1,374,292
Testing rows:  689,950
Positive test rate: 21.7981%
Baseline PR AUC: 0.2180
Model PR AUC:    0.3199
ROC AUC:         0.6519

Confusion matrix:
[[331553 208001]
 [ 58148  92248]]

Classification report:
              precision    recall  f1-score   support

           0     0.8508    0.6145    0.7136    539554
           1     0.3072    0.6134    0.4094    150396

    accuracy 

In [10]:
model_comparison = pd.DataFrame([
    {
        "model": cancelled_results["name"],
        "positive_rate": cancelled_results["test_prevalence"],
        "pr_auc": cancelled_results["pr_auc"],
        "roc_auc": cancelled_results["roc_auc"]
    },
    {
        "model": delayed_results["name"],
        "positive_rate": delayed_results["test_prevalence"],
        "pr_auc": delayed_results["pr_auc"],
        "roc_auc": delayed_results["roc_auc"]
    },
    {
        "model": diverted_results["name"],
        "positive_rate": diverted_results["test_prevalence"],
        "pr_auc": diverted_results["pr_auc"],
        "roc_auc": diverted_results["roc_auc"]
    }
])

model_comparison

,model,positive_rate,pr_auc,roc_auc
0,Cancellation model,0.014584,0.024283,0.634932
1,Departure-delay model,0.217981,0.319851,0.651929
2,Diversion model,0.002644,0.006123,0.669827
